
# Lab : GenAI Cortex Analyst

📚  In this lab you will learn and practice the following:

❄️ Understanding what Cortex Analyst is and how it translates natural language to SQL

❄️ Creating and exploring semantic views

❄️ Understanding verified queries and how they improve accuracy

❄️ Using Cortex Analyst in Snowsight to ask business questions

❄️ Understanding Routing Mode and how semantic SQL works

📌 **Note**:

Due to **regional workload spikes**, there may be **latency** with some of the steps. In the real world, for consistent performance, customers can explore [**Provisioned Throughput**](https://docs.snowflake.com/en/user-guide/snowflake-cortex/provisioned-throughput).

If you find your queries are running for more than 5 minutes, cancel and come back and try them later.

If you find models that are deprecated, use CoCo to help you fix the issue by selecting a suitable model.


---

### 🤖 Use CoCo as you go!

> **💡 TIP 1**: Use CoCo to explain complex SQL statements. Select any query and ask *"Explain this SQL"* to get a plain-language breakdown of what it does.
>
> **💡 TIP 2**: Want to learn more about any feature? Ask CoCo *"What does [feature name] do?"* to get more details and examples.
>
> **💡 TIP 3**: If you encounter a deprecated model error, ask CoCo *"Replace deprecated models in this notebook with current similar low-cost alternatives"* and it will fix them for you.

## Connect to a Service

Before running cells in this notebook, you must connect to a compute service.

**First time (create a new service):**
1. Click the **Connect** button at the top of this notebook
2. Click **Create Service** — a default name like `{{user}}_SERVICE1` will be suggested
3. Click **Service Settings** and select `ALLOW_ALL_EAI` as the external access integration
4. Leave other settings as default and click **Create**
5. Wait for the service to reach a **READY** state

**Returning (service already exists):**
1. Click the **Connect** button
2. Select your existing service from the list

Once connected, you can run Python and SQL cells interactively.

## Introduction

Cortex Analyst is a fully managed, LLM-powered feature of Snowflake Cortex that lets you ask business questions in natural language and get answers from your structured data without writing SQL.

It uses a semantic model (or semantic view) as a bridge between business language and your underlying tables, handling all the complexity of text-to-SQL generation behind the scenes.

## Getting Started

To use Cortex Analyst effectively, you need:

1. **A clear business challenge** - define what questions need answering
2. **A quality dataset** - accurate, complete, and relevant data
3. **A semantic model** - maps business terms to your tables, columns, and relationships
4. **Verified queries** - pre-validated SQL for common business questions

### Business context

At Travelbug, business users want to ask questions about bookings, activities, and reviews in natural language. The data team has prepared a semantic model that translates their business terminology into the underlying data structures.


📌 **Note:** 

* The accuracy of the answers provided by Cortex Analyst heavily depends on the dataset you supply and the semantic model  provided to Cortex Analyst. There may be instances where it doesn't fully understand your business question, resulting in either no response or an inaccurate one. In such cases, it’s essential to provide feedback to the data teams so they can update the semantic model to better align with the business requirements.
 

Here are some typical business questions at Travelbug:

- What is the total number of reviews for each activity name?
- What is the total price for bookings with a 'Completed' payment status?
- What is the booking status for all activities that occurred in the last month?
- What is the average activity price for activities in each location for the last quarter?

### What LLMs power Cortex Analyst?

Cortex Analyst is a sophisticated agentic AI system powered by industry-leading models that run securely inside Snowflake Cortex. At runtime, it selects the best combination of models to ensure the highest accuracy and performance for each query.

By default, Cortex Analyst uses Snowflake-hosted LLMs from **Mistral** and **Meta**, ensuring that no data (including metadata or prompts) leaves Snowflake's governance boundary. As LLMs evolve, Snowflake may add more models to further improve performance and accuracy.

Your role must be granted `SNOWFLAKE.CORTEX_USER` or `SNOWFLAKE.CORTEX_ANALYST_USER` to use Cortex Analyst.



### Defining a semantic model

Cortex Analyst uses a **semantic view** created directly in Snowflake to understand your business context: domain-specific terms, synonyms, default aggregations, verified queries, and join relationships.

For Travelbug, we've prepared a semantic view joining the traveler, booking, activity, and review tables.

📌 **Semantic Views** are now the recommended approach for new implementations. See: [Semantic Views Overview](https://docs.snowflake.com/en/user-guide/views-semantic/overview)

Legacy semantic model YAML files (stored on stages) are still supported for backward compatibility. See: [Semantic View YAML Spec](https://docs.snowflake.com/en/user-guide/views-semantic/semantic-view-yaml-spec)

### Semantic view reference

- [Semantic Views Documentation](https://docs.snowflake.com/en/user-guide/views-semantic/semantic-view-yaml-spec)

### Setup your current role, database, schema and warehouse.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
user = session.get_current_user().strip('"')
your_db = user + '_genai_db'
print('Your current CONTEXT information:')
print(session)

In [ ]:
%%sql -r Setup_context_sql
USE ROLE genai_role;
USE DATABASE {{user}}_genai_db;
USE SCHEMA raw;
USE WAREHOUSE {{user}}_genai_wh;
ALTER SESSION SET query_tag = '{{user}} lab - TOPIC: Cortex Analyst';
SHOW PARAMETERS LIKE 'query_tag' in session ;
SELECT "value" AS query_tag 
FROM TABLE(RESULT_SCAN(LAST_QUERY_ID())) ;

## Semantic Views
A semantic view in Snowflake is a feature that lets you add business-related meaning to your data directly within the database. It acts as a bridge, connecting the way business users think about data with how it's actually stored. 

This is achieved by defining business metrics, modeling business entities, and outlining their relationships. This creates a layer of business logic on top of the physical data, making it more intuitive and useful for making data-driven decisions. For instance, a complex metric like "net revenue" can be defined once in a semantic view, ensuring a consistent and authoritative definition across all reports and applications. This prevents inconsistencies and errors that can arise from different teams or applications calculating the same metric in different ways.

**Benefits and Use Cases**
Semantic views are particularly useful for:

**Artificial Intelligence (AI)**: They enhance the accuracy of AI applications by combining the reasoning capabilities of Large Language Models (LLMs) with rule-based definitions.

**Business Intelligence (BI)**: They provide consistent metrics and dimensions for BI tools, ensuring that everyone in the organization is working with the same understanding of the data.

**Data Analysts**: They simplify the work of data analysts by centralizing business logic, reducing the need for redundant coding and ensuring consistency.

### Semantic model components

The semantic model (or semantic view) is the critical bridge between business language and database structure. It provides:

| Component | Purpose |
|-----------|---------|
| **Logical tables** | Map business entities (customers, orders) to physical tables |
| **Dimensions** | Categorical attributes for grouping/filtering (region, status, date) |
| **Facts** | Row-level quantitative data (sale amount, quantity) |
| **Metrics** | Aggregation rules for KPIs (total revenue = SUM of amount) |
| **Relationships** | Predefined join paths between tables |
| **Verified queries** | Pre-validated SQL for common questions, used as few-shot examples |
| **Synonyms** | Maps business jargon to column names ("rev" = revenue) |
| **Custom instructions** | Domain-specific guidance for SQL generation |

### Exploring semantic views

In the **{{user}}_GENAI_DB.RESOURCES** schema you will find a semantic view that is already pre-created for you. It is a first-class object in Snowflake sitting in a database schema.

The command below is provided as an example for you to understand how the facts, dimensions, relationships, and verified queries are specified.

In [ ]:
%%sql -r Exploring_semantic_views_sql
-- Create or replace a semantic view for the Travelbug Analyst application
CREATE OR REPLACE SEMANTIC VIEW RESOURCES.TRAVELBUG_SEMANTIC_MODEL

  -- Declare the source tables and their primary keys
  TABLES (
    ACTIVITY AS TRANSFORMED.ACTIVITY PRIMARY KEY (ACTIVITY_ID), -- Activities offered
    BOOKING AS TRANSFORMED.BOOKING PRIMARY KEY (BOOKING_ID),     -- Bookings made by travelers
    REVIEW AS TRANSFORMED.REVIEW PRIMARY KEY (REVIEW_ID),        -- Reviews submitted
    TRAVELER AS TRANSFORMED.TRAVELER PRIMARY KEY (TRAVELER_ID) -- Traveler profiles
  )

  -- Define relationships between tables for join logic
  RELATIONSHIPS (
    BOOKING (ACTIVITY_ID) REFERENCES ACTIVITY,       -- Each booking is linked to an activity
    BOOKING (TRAVELER_ID) REFERENCES TRAVELER,     -- Each booking is made by a traveler
    REVIEW (BOOKING_ID) REFERENCES BOOKING           -- Each review is tied to a booking
  )

  -- Declare measurable numeric fields (facts) for analysis
  FACTS (
    ACTIVITY.capacity AS CAPACITY                    -- Max participants per activity
        COMMENT='The maximum number of participants for an activity.',
    ACTIVITY.duration AS DURATION                    -- Duration in hours
        COMMENT='The length of time required to complete the activity, in hours.',
    ACTIVITY.price AS PRICE                          -- Price of the activity
        COMMENT='The amount paid or charged for a particular activity or service.',
    BOOKING.total_price AS TOTAL_PRICE               -- Total booking cost
        COMMENT='The total cost of a booking, including all applicable fees and taxes.'
  )

  -- Define descriptive fields (dimensions) for filtering and grouping
  DIMENSIONS (
    ACTIVITY.activity_id AS ACTIVITY_ID              -- Unique activity ID
        COMMENT='Unique identifier for a specific activity or task performed within a process or workflow.',
    ACTIVITY.activity_name AS NAME                   -- Name of the activity
        COMMENT='Name of the activity. Could be Kayaking Adventure, Rock Climbing, Sunset Boat Cruise, etc.',
    ACTIVITY.activity_description AS DESCRIPTION     -- Description of the activity
        COMMENT='A brief description of various activities offered.',
    ACTIVITY.activity_location AS LOCATION           -- Location of the activity
        COMMENT='The physical location where an activity takes place.',
    BOOKING.booking_id AS BOOKING_ID                 -- Unique booking ID
        COMMENT='Unique identifier for each booking.',
    BOOKING.booking_date AS BOOKING_DATE             -- Date the booking was made
        COMMENT='Date on which a booking was made.',
    BOOKING.activity_date AS ACTIVITY_DATE           -- Date the activity occurred
        COMMENT='Date on which a booking activity took place.',
    BOOKING.booking_status AS BOOKING_STATUS         -- Status of the booking
        COMMENT='The current status of a booking (e.g., Confirmed, Canceled).',
    BOOKING.payment_status AS PAYMENT_STATUS         -- Payment status
        COMMENT='Indicates if the payment has been successfully made (e.g., Paid, Refunded).',
    BOOKING.traveler_id AS TRAVELER_ID             -- Traveler who made the booking
        COMMENT='Unique identifier of the traveler who made the booking.',
    REVIEW.review_id AS REVIEW_ID                    -- Unique review ID
        COMMENT='Unique identifier for each review in the system.',
    REVIEW.review_date AS REVIEW_DATE                -- Date the review was submitted
        COMMENT='The date on which a review was submitted or posted.',
    REVIEW.review_text AS REVIEW_TEXT                -- Review content
        COMMENT='Customer reviews and feedback about their experiences.',
    TRAVELER.traveler_id AS TRAVELER_ID           -- Unique traveler ID
        COMMENT='Unique identifier for each traveler.',
    TRAVELER.traveler_name AS TRAVELER_NAME       -- Traveler’s full name
        COMMENT='The full name of the traveler.',
    TRAVELER.traveler_email AS EMAIL               -- Traveler’s email
        COMMENT='The email address of the traveler.',
    TRAVELER.traveler_address AS ADDRESS           -- Traveler’s address
        COMMENT='Physical address of the traveler.',
    TRAVELER.date_of_birth AS DATE_OF_BIRTH         -- Traveler’s birth date
        COMMENT='Date of birth of the traveler.',
    TRAVELER.phone_number AS PHONE_NUMBER           -- Traveler’s phone number
        COMMENT='The phone number of the traveler.'
  )

  -- Define calculated metrics using aggregate functions
  METRICS (
    BOOKING.total_revenue AS SUM(BOOKING.TOTAL_PRICE) -- Total revenue from bookings
        COMMENT='Total revenue generated from all bookings.',
    BOOKING.average_booking_value AS AVG(BOOKING.TOTAL_PRICE) -- Average booking value
        COMMENT='Average value of bookings across all transactions.',
    BOOKING.total_bookings AS COUNT(BOOKING.BOOKING_ID) -- Total number of bookings
        COMMENT='Total number of bookings made.',
    ACTIVITY.average_duration AS AVG(ACTIVITY.DURATION) -- Average activity duration
        COMMENT='Average duration of activities in hours.',
    ACTIVITY.average_capacity AS AVG(ACTIVITY.CAPACITY) -- Average activity capacity
        COMMENT='Average capacity across all activities.',
    ACTIVITY.average_price AS AVG(ACTIVITY.PRICE)       -- Average activity price
        COMMENT='Average price of activities.',
    ACTIVITY.activity_count AS COUNT(ACTIVITY.ACTIVITY_ID) -- Total number of activities
        COMMENT='Total number of unique activities available.',
    REVIEW.total_reviews AS COUNT(REVIEW.REVIEW_ID)     -- Total number of reviews
        COMMENT='Total number of reviews submitted.',
    TRAVELER.traveler_count AS COUNT(DISTINCT TRAVELER.TRAVELER_ID) -- Unique travelers
        COMMENT='Total number of unique travelers.'
  
)
-- Add a description for the semantic view
COMMENT='This semantic model for the Travelbug Analyst application focuses on the traveler, booking, activity, and review tables in the transformed schema. It is designed to help answer business questions raised by analysts and includes comprehensive metrics for performance analysis.'
;


In [ ]:
%%sql -r Exploring_semantic_views_results_sql
SHOW SEMANTIC VIEWS IN SCHEMA RESOURCES;

In [ ]:
%%sql -r Exploring_semantic_views_describe_sql
DESCRIBE SEMANTIC VIEW RESOURCES.TRAVELBUG_SEMANTIC_MODEL;

In [ ]:
%%sql -r Exploring_semantic_views_show_dimensions_sql
USE SCHEMA RESOURCES;
SHOW SEMANTIC DIMENSIONS;

In [ ]:
%%sql -r Exploring_semantic_views_get_ddl_sql
SELECT GET_DDL('SEMANTIC_VIEW', '{{user}}_genai_db.resources.travelbug_semantic_model');

In [ ]:
# Format output from the previous cell
df = Exploring_semantic_views_get_ddl_sql.to_pandas()

ddl_text = df.iloc[0, 0]
print(ddl_text)

## Using Cortex Analyst in Snowsight

Now that we have explored the semantic model and semantic view, let's use Cortex Analyst directly in Snowsight to ask business questions.

**Steps:**

1. **Duplicate your browser tab.** Right-click the Snowsight tab and select **Duplicate** (or Ctrl+Shift+T / Cmd+Shift+T) so you can keep this notebook open for reference.

2. In the new tab, navigate to **AI & ML  > AI Studio >  Analyst** from the left navigation menu.

3. You will be prompted to select a semantic view. Set the following:
   - **Database**: `<your_user>_GENAI_DB`
   - **Schema**: `RESOURCES`
   - **Semantic View**: `TRAVELBUG_SEMANTIC_MODEL`

4. Once the semantic view is loaded, click on **Playground** to open the interactive query interface.

5. In the Playground, you will see **suggested questions** based on your semantic view's verified queries (those with `USE_AS_ONBOARDING_QUESTION = TRUE`). Click on any of these suggested questions to get started quickly, or type your own question in natural language. Try some of the business questions from earlier in this lab:
   - *What is the total number of reviews for each activity name?*
   - *What is the total price for bookings with a 'Completed' payment status?*
   - *What is the booking status for all activities that occurred in the last month?*
   - *What is the average activity price for activities in each location for the last quarter?*

6. Observe how Cortex Analyst generates SQL and returns results based on your semantic view definition.

### Querying the semantic view directly

You can query a semantic view directly using SQL with the `SEMANTIC_VIEW` clause in the `FROM` statement. This lets you select dimensions and metrics by their logical names without needing to know the underlying table structure or write joins manually.

The syntax is:
```sql
SELECT * FROM SEMANTIC_VIEW(
  <semantic_view_name>
  DIMENSIONS <table>.<dimension>, ...
  METRICS <table>.<metric>, ...
)
```

See: [Querying Semantic Views](https://docs.snowflake.com/en/user-guide/views-semantic/querying)

In [ ]:
%%sql -r semantic_view_query
-- Total reviews per activity name
SELECT * FROM SEMANTIC_VIEW(
  RESOURCES.TRAVELBUG_SEMANTIC_MODEL
  DIMENSIONS ACTIVITY.ACTIVITY_NAME
  METRICS REVIEW.TOTAL_REVIEWS
)
ORDER BY TOTAL_REVIEWS DESC;

In [ ]:
%%sql -r revenue_by_location
-- Total revenue and average booking value by activity location
SELECT * FROM SEMANTIC_VIEW(
  RESOURCES.TRAVELBUG_SEMANTIC_MODEL
  DIMENSIONS ACTIVITY.ACTIVITY_LOCATION
  METRICS BOOKING.TOTAL_REVENUE, BOOKING.AVERAGE_BOOKING_VALUE
)
ORDER BY TOTAL_REVENUE DESC;

In [ ]:
%%sql -r bookings_by_status
-- Booking count by booking status
SELECT * FROM SEMANTIC_VIEW(
  RESOURCES.TRAVELBUG_SEMANTIC_MODEL
  DIMENSIONS BOOKING.BOOKING_STATUS
  METRICS BOOKING.TOTAL_BOOKINGS
)
ORDER BY TOTAL_BOOKINGS DESC;

In [ ]:
%%sql -r price_duration_by_location
-- Average price and average duration by activity location
SELECT * FROM SEMANTIC_VIEW(
  RESOURCES.TRAVELBUG_SEMANTIC_MODEL
  DIMENSIONS ACTIVITY.ACTIVITY_LOCATION
  METRICS ACTIVITY.AVERAGE_PRICE, ACTIVITY.AVERAGE_DURATION
)
ORDER BY AVERAGE_PRICE DESC;

### Usage guidelines for Cortex Analyst

- Use the semantic view generator in Snowsight (AI & ML Studio > Cortex Analyst).
- Provide verified queries for high-precision answers.
- Choose a focused dataset for a particular business domain.
- Govern access to the semantic view using standard Snowflake roles and grants.

### Verified queries (VQR)

Verified queries are pre-validated SQL that improve accuracy for common business questions. Cortex Analyst uses them as few-shot examples when answering similar questions.

A verified query includes a natural language question and the correct SQL to answer it. The SQL must reference the **logical names** defined in your semantic view, not the physical column names.

Here is the syntax for adding verified queries to a semantic view:

```sql
ALTER SEMANTIC VIEW RESOURCES.TRAVELBUG_SEMANTIC_MODEL
  ADD VERIFIED QUERIES (
    QUERY total_reviews_by_activity
      QUESTION = 'What is the total number of reviews for each activity name?'
      SQL = '
        SELECT NAME, COUNT(REVIEW_ID) AS total_reviews
        FROM __ACTIVITY
        JOIN __BOOKING ON __ACTIVITY.ACTIVITY_ID = __BOOKING.ACTIVITY_ID
        JOIN __REVIEW ON __BOOKING.BOOKING_ID = __REVIEW.BOOKING_ID
        GROUP BY NAME
        ORDER BY total_reviews DESC
      '
      VERIFIED_BY = 'data_team'
      USE_AS_ONBOARDING_QUESTION = TRUE
  );
```

Key points:
- Table names are prefixed with double underscores (`__ACTIVITY`) to reference logical tables
- Column names use the logical names from your DIMENSIONS/FACTS/METRICS definitions
- `USE_AS_ONBOARDING_QUESTION = TRUE` surfaces the question as a suggested starting prompt
- Invalid or inaccurate verified queries will negatively impact accuracy

### Routing Mode

When you use Cortex Analyst with a semantic view, it automatically uses **Routing Mode**. This means it tries to generate semantic SQL first:

```sql
SELECT *
FROM SEMANTIC_VIEW(
  travelbug_semantic_model
  DIMENSIONS activity.name
  METRICS review.total_reviews
)
ORDER BY total_reviews DESC;
```

With Routing Mode:
- **Metrics, joins, and filters follow governed definitions** from the semantic view
- If the semantic view cannot satisfy a question, Cortex Analyst automatically **falls back to standard SQL** on the physical tables
- No configuration is needed. It is the default behavior when working with semantic views

This is why you may see `SELECT FROM SEMANTIC_VIEW(...)` syntax in the generated SQL output when testing in Snowsight.

## 🎯 Challenge Questions

Test your understanding of the concepts covered in this lab.

In [ ]:
from snowflake.snowpark.context import get_active_session
from IPython.display import display, HTML

session = get_active_session()

quiz_data = [
    {"q": "What is the purpose of a semantic model in Cortex Analyst?", "options": ["A) It stores the actual data for analysis", "B) A semantic model defines the business meaning and relationships of data for natural language queries", "C) It replaces the need for SQL queries entirely", "D) It is only used for data visualization"], "hash": "9b9f8714ec6b04478a6ca7511d308ec1"},
    {"q": "What are Verified Queries (VQRs) in Cortex Analyst?", "options": ["A) Queries that have been automatically generated by AI", "B) Queries that must be approved by an administrator", "C) Verified queries (VQRs) are pre-validated SQL examples that improve accuracy for common questions", "D) Queries that have been executed at least once"], "hash": "2a4f64b0b56be48a5d32997518abcf75"},
    {"q": "What is a Semantic View in Snowflake?", "options": ["A) A regular database view with added comments", "B) A view that only shows semantic data types", "C) A materialized view for faster queries", "D) Semantic views combine the semantic model definition with database object metadata"], "hash": "9baccd4485b920e2b7125ebd10c52fe7"},
    {"q": "What role do custom instructions play in a semantic model?", "options": ["A) Custom instructions provide domain-specific guidance to help the model understand business context", "B) They define the SQL syntax to use", "C) They specify which users can access the model", "D) They control the response format only"], "hash": "992f1df348918aa365e7098d2003e643"},
    {"q": "Why are join relationships important in a semantic model?", "options": ["A) They improve query performance by caching data", "B) Join relationships in semantic models define how tables relate to enable multi-table queries", "C) They are required for all Cortex Analyst queries", "D) They automatically create foreign keys in the database"], "hash": "c3e0d0e6d8e962ad560d7c6ab3682ccb"},
]

results_map = {}
for qi, item in enumerate(quiz_data):
    results_map[qi] = {}
    for opt in item["options"]:
        letter = opt[0]
        escaped_opt = opt.replace("'", "''")
        result = session.sql(f"CALL genai_db.resources.quiz_temp('{item['hash']}', '{escaped_opt}', 'False')").collect()
        feedback = result[0][0]
        is_correct = 'Correct' in feedback or '✅' in feedback
        results_map[qi][letter] = (feedback, is_correct)

html = """<style>
.cq { margin: 20px 0; padding: 16px; border: 1px solid #d0d0d0; border-radius: 10px; background: #fafafa; font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, "Helvetica Neue", Arial, sans-serif; font-size: 14px; }
.cq h4 { font-family: inherit; }
.cq input[type="radio"] { display: none; }
.cq .lbl { display: block; padding: 8px 12px; border-radius: 6px; cursor: pointer; font-family: inherit; }
.cq .lbl:hover { background: #e8f0fe; }
.cq input[type="radio"]:checked + .lbl { border-color: #1a73e8; background: #e8f0fe; font-weight: 600; }
.cq .fb { display: none; padding: 6px 12px; margin-top: 2px; border-radius: 4px; font-weight: 600; font-family: inherit; }
.cq input[type="radio"]:checked + .lbl + .fb { display: block; }
.cq .fb.ok { background: #e6f4ea; color: #1e7e34; }
.cq .fb.no { background: #fce8e6; color: #c62828; }
</style>"""

for qi, item in enumerate(quiz_data):
    html += f'<div class="cq"><h4>Q{qi+1}: {item["q"]}</h4>'
    for opt in item["options"]:
        letter = opt[0]
        feedback, is_correct = results_map[qi][letter]
        css_class = 'ok' if is_correct else 'no'
        uid = f'cq{qi}_{letter}'
        html += f'<div class="opt"><input type="radio" name="cq{qi}" id="{uid}">'
        html += f'<label class="lbl" for="{uid}">{opt}</label>'
        html += f'<div class="fb {css_class}">{letter}) {feedback}</div></div>'
    html += '</div>'

display(HTML(html))

## Key Takeaways

❄️ Cortex Analyst translates natural language questions into SQL using semantic views as the bridge between business language and database structure.

❄️ Semantic views define logical tables, dimensions, facts, metrics, and relationships that guide accurate SQL generation.

❄️ Verified queries (VQR) improve accuracy by providing pre-validated SQL examples for common business questions.

❄️ Routing Mode automatically generates semantic SQL (`SELECT FROM SEMANTIC_VIEW(...)`) and falls back to standard SQL when needed.

❄️ Data quality and a well-defined semantic model are key to getting high-precision answers. This is an iterative process.